# FDA NDC to WHO ATC

Locked extract. Prescription products matched to the WHO ATC substance name at 85.3%.

| File | Rows | What it is |
| --- | --- | --- |
| ndc_matched.csv | 47,795 | Hit. Product plus ATC code and the three class levels |
| ndc_fallout.csv | 8,240 | Miss, with a reason |
| ndc_class_roll.csv | 80 | Product count by anatomical and therapeutic class |

Folder: `C:\NDC API Project`

Run the build cells only if the frames are not already in memory. The join cell is locked. The review cells at the bottom only read the files.


In [ ]:
# Libraries for the FDA NDC pull
# FETCH - full FDA NDC directory from the download file
# The API page loop stops at 26,000. This file is the whole catalog.
# One zip, about 27 MB, 138,310 products. Same shape as the API results.
# Run once. Later runs reuse the zip if it is already on disk.
# SOURCE - write the catalog to CSV before any explode or drop.
# Nested lists stay as text. Grain is still one row per product.
# requests: call the API
# pandas: build the tables
# json: read and write the raw response

import zipfile
from pathlib import Path
import requests
import pandas as pd
import json

url = "https://download.open.fda.gov/drug/ndc/drug-ndc-0001-of-0001.json.zip"
folder = Path(r"C:\NDC API Project")
folder.mkdir(parents=True, exist_ok=True)
zip_path = folder / "drug-ndc.json.zip"
json_name = "drug-ndc-0001-of-0001.json"
source_path = folder / "ndc_source_before_processing.csv"

if not zip_path.exists():
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    zip_path.write_bytes(response.content)
    print("downloaded", zip_path)

with zipfile.ZipFile(zip_path) as z:
    with z.open(json_name) as f:
        data = json.load(f)

results = data["results"]
print("done", len(results))

def cell(v):
    if isinstance(v, (list, dict)):
        return json.dumps(v, ensure_ascii=False)
    return v

source = pd.DataFrame([{k: cell(v) for k, v in drug.items()} for drug in results])
source.to_csv(source_path, index=False)
print("wrote", source_path, "rows", len(source))


## Grain

| Frame | Grain | Joins on |
| --- | --- | --- |
| product | one row per product_ndc | |
| package | one row per ndc11 | product_ndc |
| ingredient | one row per product + ingredient | product_ndc |
| drug_class | one row per product + class | product_ndc |
| route | one row per product + route | product_ndc |
| lookup | one row per product + openFDA field + sequence | product_ndc |

Product is the parent. Package is the 11-digit billing key. Ingredient, class, and route hang off the product and stay in their own files.


In [ ]:
# BUILD - product dimension
# Grain: one row per product_ndc (labeler + product code).
# Packages, ingredients, and classes are other frames. They do not belong here.

rows = []
for drug in results:
    product_ndc = drug.get("product_ndc")
    if not product_ndc or "-" not in product_ndc:
        continue
    labeler, product_code = product_ndc.split("-", 1)
    rows.append({
        "product_ndc": product_ndc,
        "labeler_code": labeler.zfill(5),
        "product_code": product_code.zfill(4),
        "brand_name": drug.get("brand_name"),
        "generic_name": drug.get("generic_name"),
        "labeler_name": drug.get("labeler_name"),
        "dosage_form": drug.get("dosage_form"),
        "product_type": drug.get("product_type"),
        "marketing_category": drug.get("marketing_category"),
        "application_number": drug.get("application_number"),
        "dea_schedule": drug.get("dea_schedule"),
        "finished": drug.get("finished"),
        "marketing_start_date": drug.get("marketing_start_date"),
        "marketing_end_date": drug.get("marketing_end_date"),
        "listing_expiration_date": drug.get("listing_expiration_date"),
    })

product = pd.DataFrame(rows).drop_duplicates(subset=["product_ndc"])
print("product rows", len(product))
print("unique product_ndc", product["product_ndc"].nunique())
product.head(3)


In [ ]:
# BUILD - package
# Grain: one row per package_ndc, stored as ndc11.
# product_ndc is the foreign key. Product attributes stay on product.

def ndc11(product_ndc, package_ndc):
    labeler, product_code = product_ndc.split("-", 1)
    package_code = package_ndc.split("-")[-1]
    return labeler.zfill(5) + product_code.zfill(4) + package_code.zfill(2)

rows = []
for drug in results:
    product_ndc = drug.get("product_ndc")
    if not product_ndc:
        continue
    for pack in drug.get("packaging") or []:
        package_ndc = pack.get("package_ndc")
        if not package_ndc:
            continue
        rows.append({
            "ndc11": ndc11(product_ndc, package_ndc),
            "package_ndc": package_ndc,
            "product_ndc": product_ndc,
            "description": pack.get("description"),
            "sample": pack.get("sample"),
            "marketing_start_date": pack.get("marketing_start_date"),
            "marketing_end_date": pack.get("marketing_end_date"),
        })

package = pd.DataFrame(rows).drop_duplicates(subset=["ndc11"])
print("package rows", len(package))
print("unique ndc11", package["ndc11"].nunique())
print("products with a package", package["product_ndc"].nunique())
package.head(3)


In [ ]:
# BUILD - active ingredient
# Grain: one row per product_ndc + ingredient name.
# Strength belongs here. A combo product becomes two or more rows.

rows = []
for drug in results:
    product_ndc = drug.get("product_ndc")
    if not product_ndc:
        continue
    for ing in drug.get("active_ingredients") or []:
        name = ing.get("name")
        if not name:
            continue
        rows.append({
            "product_ndc": product_ndc,
            "ingredient_name": name.strip().upper(),
            "strength": ing.get("strength"),
        })

ingredient = pd.DataFrame(rows).drop_duplicates(
    subset=["product_ndc", "ingredient_name", "strength"]
)
print("ingredient rows", len(ingredient))
print("products with an ingredient", ingredient["product_ndc"].nunique())
print("unique ingredient names", ingredient["ingredient_name"].nunique())
ingredient.head(3)


In [ ]:
# BUILD - drug class
# Grain: one row per product_ndc + class.
# Bulk file keeps each class type in its own list, not one pharm_class field.

CLASS_FIELDS = {
    "pharm_class_epc": "EPC",
    "pharm_class_moa": "MOA",
    "pharm_class_cs": "CS",
    "pharm_class_pe": "PE",
    "pharm_class": None,
}

TYPE_NAME = {
    "EPC": "Established Pharmacologic Class",
    "MOA": "Mechanism of Action",
    "MoA": "Mechanism of Action",
    "CS": "Chemical Structure",
    "PE": "Physiologic Effect",
}

rows = []
for drug in results:
    product_ndc = drug.get("product_ndc")
    openfda = drug.get("openfda") or {}
    if not product_ndc:
        continue
    for field, class_type in CLASS_FIELDS.items():
        for raw in openfda.get(field) or []:
            if not raw:
                continue
            if " [" in raw:
                class_name, parsed = raw.split(" [", 1)
                class_type = parsed.rstrip("]")
            else:
                class_name = raw
            rows.append({
                "product_ndc": product_ndc,
                "class_type": class_type,
                "class_type_name": TYPE_NAME.get(class_type),
                "class_name": class_name.strip().upper(),
            })

drug_class = pd.DataFrame(rows)
if drug_class.empty:
    print("no class rows")
else:
    drug_class = drug_class[drug_class["class_type"] != "Chemical/Ingredient"]
    drug_class = drug_class.drop_duplicates(
        subset=["product_ndc", "class_type", "class_name"]
    )
    print("class rows", len(drug_class))
    print("products with a class", drug_class["product_ndc"].nunique())
    print(drug_class["class_type_name"].value_counts(dropna=False))
    display(drug_class.head(3))


In [ ]:
# BUILD - route
# Grain: one row per product_ndc + route.
# A product can be oral and intravenous. That is two rows, not one.

rows = []
for drug in results:
    product_ndc = drug.get("product_ndc")
    if not product_ndc:
        continue
    for route in drug.get("route") or []:
        if not route:
            continue
        rows.append({
            "product_ndc": product_ndc,
            "route": route.strip().upper(),
        })

route = pd.DataFrame(rows).drop_duplicates(subset=["product_ndc", "route"])
print("route rows", len(route))
print("products with a route", route["product_ndc"].nunique())
print(route["route"].value_counts().head(8))
route.head(3)


In [ ]:
# ndc_openfda_lookup.csv
# Reference extract of the openfda annotation block. Not a normalized model.
# Source: openFDA drug NDC directory. Nested openfda dict stays on the source CSV.
# Grain: one row per product_ndc + field + value. Lists become multiple rows.
# value_seq keeps list order. value is always text, including booleans.
# Join back to the product file on product_ndc. Filter on field when you need a slice.
# Do not explode this into the product file. Normalize a slice only if you build on it.
# Fields seen: application_number, brand_name, generic_name, manufacturer_name,
# product_ndc, product_type, package_ndc, route, substance_name, rxcui, spl_id,
# spl_set_id, unii, nui, pharm_class_moa, pharm_class_epc, pharm_class_cs,
# pharm_class_pe, upc, is_original_packager.

rows = []
for drug in results:
    product_ndc = drug.get("product_ndc")
    openfda = drug.get("openfda") or {}
    for field, raw in openfda.items():
        values = raw if isinstance(raw, list) else [raw]
        for i, value in enumerate(values, start=1):
            if value is None:
                continue
            rows.append({
                "product_ndc": product_ndc,
                "field": field,
                "value_seq": i,
                "value": value,
            })

lookup = pd.DataFrame(rows)
lookup_path = folder / "ndc_openfda_lookup.csv"
lookup.to_csv(lookup_path, index=False)
print("wrote", lookup_path, "rows", len(lookup), "fields", lookup["field"].nunique())


In [ ]:
# WRITE - six frames, one grain each
# product: one row per product_ndc
# package: one row per ndc11
# ingredient: one row per product_ndc + ingredient
# drug_class: one row per product_ndc + class
# route: one row per product_ndc + route
# openfda lookup: one row per product_ndc + field + value

product.to_csv(folder / "product.csv", index=False)
package.to_csv(folder / "package.csv", index=False)
ingredient.to_csv(folder / "ingredient.csv", index=False)
drug_class.to_csv(folder / "drug_class.csv", index=False)
route.to_csv(folder / "route.csv", index=False)
lookup.to_csv(folder / "ndc_openfda_lookup.csv", index=False)

for name, frame in {
    "product": product,
    "package": package,
    "ingredient": ingredient,
    "drug_class": drug_class,
    "route": route,
    "ndc_openfda_lookup": lookup,
}.items():
    print(name, len(frame))


## ATC join

WHO file is the substance list, level 5, seven-character code. The join is on the cleaned generic name, not the NDC. The 8,240 misses are combos, US names the file does not carry, and comma-split ingredients. Leave them in the fallout file.


In [ ]:
# WHO ATC-DDD hierarchy, scraped 2026-04-25 by fabkury/atcd.
# This is the class tree, not the combinations file. No brand names in here.
# atc_name is the generic. Level is the length of the code: 1, 3, 4, 5, 7.
import pandas as pd

atc = pd.read_csv(r"C:\NDC API Project\WHO.ATC-DDD.2026-04-25.csv")
atc["atc_code"] = atc["atc_code"].astype("string").str.strip()
atc["atc_name"] = atc["atc_name"].astype("string").str.strip()
atc["level"] = atc["atc_code"].str.len().map({1: 1, 3: 2, 4: 3, 5: 4, 7: 5})

# Expect about 7,536 rows and columns atc_code, atc_name, ddd, uom, adm_r, note.
# Level 5 should be near 5,678. That is the substance grain you join to FDA.
print(atc.shape)
print(atc.columns.tolist())
print(atc["level"].value_counts().sort_index())
atc.head()


In [ ]:
# LOCKED JOIN. 85.3%. Do not edit.
# 56,035 prescription products, 47,795 matched, 8,240 fallout.
# Alias passes after this rewrote names that already matched.

import re
import pandas as pd

salts = sorted({
    "dihydrochloride", "hydrochloride", "hydrobromide", "mesylate", "besylate",
    "tosylate", "fumarate", "maleate", "tartrate", "bitartrate", "succinate",
    "citrate", "acetate", "phosphate", "sulfate", "sulphate", "nitrate",
    "hcl", "furoate", "sodium", "potassium", "monohydrate", "dihydrate",
    "trihydrate", "anhydrous", "tromethamine", "hyclate", "dimesylate",
    "calcium", "pamoate", "carbonate", "ethyl",
    "magnesium", "chloride", "propionate", "acetonide", "benzoate", "cypionate",
    "saccharate", "aspartate",
}, key=len, reverse=True)
forms = sorted({
    "tablet", "tablets", "capsule", "capsules", "injection", "solution",
    "suspension", "cream", "ointment", "gel", "patch", "spray", "syrup",
    "extended-release", "delayed-release", "usp",
}, key=len, reverse=True)
salt_re = re.compile(r"\b(?:" + "|".join(salts) + r")\b")
form_re = re.compile(r"\b(?:" + "|".join(forms) + r")\b")

spelling = {
    "cephalexin": "cefalexin",
    "chlorthalidone": "chlortalidone",
    "albuterol": "salbutamol",
    "acetaminophen": "paracetamol",
    "meperidine": "pethidine",
    "rifampin": "rifampicin",
    "glyburide": "glibenclamide",
    "acyclovir": "aciclovir",
    "valacyclovir": "valaciclovir",
    "dicyclomine": "dicycloverine",
    "glycopyrrolate": "glycopyrronium",
    "indomethacin": "indometacin",
    "mesalamine": "mesalazine",
    "penicillin v": "phenoxymethylpenicillin",
}
alias = {
    "divalproex": "valproic acid",
    "mycophenolate mofetil": "mycophenolic acid",
    "amoxicillin / clavulanate": "amoxicillin / beta-lactamase inhibitor",
    "dextrose": "glucose",
    "thyroid / porcine": "thyroid gland preparations",
    "niacin": "nicotinic acid",
    "vasopressin": "argipressin",
    "urea": "carbamide",
    "carbidopa / levodopa": "levodopa and decarboxylase inhibitor",
    "triamterene / hydrochlorothiazide": "hydrochlorothiazide and potassium-sparing agents",
    "ceftazidime / avibactam": "ceftazidime and beta-lactamase inhibitor",
    "tobramycin / dexamethasone": "dexamethasone and antiinfectives",
    "nitroglycerin": "glyceryl trinitrate",
    "succinylcholine": "suxamethonium",
    "succinylcholine chloride": "suxamethonium",
    "cosyntropin": "tetracosactide",
    "hydrocodone / paracetamol": "hydrocodone and paracetamol",
    "oxycodone / paracetamol": "oxycodone and paracetamol",
    "losartan / hydrochlorothiazide": "losartan and diuretics",
    "valsartan / hydrochlorothiazide": "valsartan and diuretics",
    "paracetamol / codeine": "codeine and paracetamol",
    "codeine / paracetamol": "codeine and paracetamol",
    "buprenorphine / naloxone": "buprenorphine and naloxone",
}

substance = atc[atc["atc_code"].str.len() == 7].copy()
substance["join_name"] = substance["atc_name"].str.strip().str.lower()
atc_name = substance.drop_duplicates("join_name")[
    ["atc_code", "atc_name", "join_name", "defined_daily_dose", "uom", "administration_route"]
]
atc_keys = set(atc_name["join_name"])
atc_keys |= {k.replace(" / ", " and ") for k in atc_keys}
atc_keys |= {k.replace(" and ", " / ") for k in atc_keys}

def spell_parts(name):
    parts = [spelling.get(p.strip(), p.strip()) for p in name.split("/") if p.strip()]
    return " / ".join(parts)

def candidates(value):
    raw = "" if pd.isna(value) else str(value).strip().lower()
    raw = raw.replace(" and ", " / ").replace(",", " / ")
    raw = form_re.sub(" ", raw)
    raw = re.sub(r"\s+", " ", raw).strip(" ,/")
    stripped = raw
    previous = None
    while previous != stripped:
        previous = stripped
        trial = salt_re.sub(" ", stripped)
        trial = re.sub(r"\s+", " ", trial).strip(" ,/")
        stripped = trial if trial else stripped
    names = []
    for name in (raw, stripped):
        name = spell_parts(name)
        name = alias.get(name, name)
        names.append(name)
        parts = [p.strip() for p in name.split("/") if p.strip()]
        if len(parts) == 2:
            names.append(" / ".join(reversed(parts)))
            names.append(" and ".join(parts))
            names.append(" and ".join(reversed(parts)))
    return list(dict.fromkeys(names))

def base_name(value):
    options = candidates(value)
    for name in options:
        key = name.replace(" and ", " / ")
        if key in atc_keys or name in atc_keys:
            return key
    return options[0].replace(" and ", " / ")

rx = product[product["product_type"].eq("HUMAN PRESCRIPTION DRUG")].copy()
rx["join_name"] = rx["generic_name"].map(base_name)
atc_name = atc_name.copy()
atc_name["join_name"] = atc_name["join_name"].str.replace(" and ", " / ", regex=False)
matched = rx.merge(atc_name, on="join_name", how="left", indicator=True)
hit = matched["_merge"].eq("both")

print("prescription products", len(rx))
print("matched", int(hit.sum()), f"{hit.mean():.1%}")
print("fallout", int((~hit).sum()))
print(matched.loc[~hit, ["product_ndc", "brand_name", "generic_name", "join_name"]].head(15).to_string(index=False))


In [ ]:
from pathlib import Path
import pandas as pd

folder = Path(r"C:\NDC API Project")

level = {
    n: atc.loc[atc["atc_code"].str.len().eq(n)]
    .drop_duplicates("atc_code")
    .set_index("atc_code")["atc_name"]
    for n in (1, 3, 5)
}

work = matched.copy()
work["anatomical_class"] = work["atc_code"].str[0].map(level[1])
work["therapeutic_class"] = work["atc_code"].str[:3].map(level[3])
work["chemical_class"] = work["atc_code"].str[:5].map(level[5])

keep = [
    "product_ndc", "brand_name", "generic_name", "labeler_name",
    "dosage_form", "product_type", "join_name",
    "atc_code", "atc_name", "anatomical_class", "therapeutic_class", "chemical_class",
]
keep = [c for c in keep if c in work.columns]

hit = work["_merge"].eq("both")
included = work.loc[hit, keep].sort_values("product_ndc")

def reason(row):
    name = "" if pd.isna(row["join_name"]) else str(row["join_name"])
    if name.strip() == "":
        return "blank_name"
    if name.endswith("-xioi") or "abparvovec" in name:
        return "gene_or_biologic"
    if name.count("/") >= 3:
        return "comma_split"
    if "/" in name:
        return "combo_not_in_who"
    return "not_in_who_file"

fallout = work.loc[~hit].copy()
fallout["reason"] = fallout.apply(reason, axis=1)
fallout = fallout[keep + ["reason"]].sort_values(["reason", "generic_name"])

roll = (
    included.dropna(subset=["therapeutic_class"])
    .groupby(["anatomical_class", "therapeutic_class"], as_index=False)
    .agg(products=("product_ndc", "nunique"))
    .sort_values("products", ascending=False)
)

included.to_csv(folder / "ndc_matched.csv", index=False)
fallout.to_csv(folder / "ndc_fallout.csv", index=False)
roll.to_csv(folder / "ndc_class_roll.csv", index=False)

print("matched", len(included))
print("fallout", len(fallout))
print(fallout["reason"].value_counts().to_string())
print("classes", len(roll))
print(roll.head(10).to_string(index=False))


## Read the extract

`ndc_matched.csv` is the file to learn. `atc_code` is the 7-character substance. `anatomical_class` is the first character. `therapeutic_class` is the first three. `chemical_class` is the first five.

Fallout reasons: `not_in_who_file` is a single drug the file does not name, `combo_not_in_who` is a pair, `comma_split` is a salt list that broke into pieces, `gene_or_biologic` is a gene therapy.


In [ ]:
# REVIEW - read the extract. Does not rebuild or overwrite.
matched_file = pd.read_csv(folder / "ndc_matched.csv", dtype=str)
fallout_file = pd.read_csv(folder / "ndc_fallout.csv", dtype=str)
roll_file = pd.read_csv(folder / "ndc_class_roll.csv", dtype=str)

print("matched", len(matched_file))
print("fallout", len(fallout_file))
print(fallout_file["reason"].value_counts().to_string())
print()
print(roll_file.head(10).to_string(index=False))


In [ ]:
# One class, then one drug. Change the two strings. Nothing is written.
klass = "ANALGESICS"
name = "oxycodone"

print(matched_file.loc[matched_file["therapeutic_class"].eq(klass),
                       ["product_ndc", "brand_name", "generic_name", "atc_code", "atc_name"]]
      .head(8).to_string(index=False))
print()
print(matched_file.loc[matched_file["generic_name"].str.contains(name, case=False, na=False),
                       ["product_ndc", "brand_name", "generic_name", "atc_code", "chemical_class"]]
      .drop_duplicates("atc_code").head(8).to_string(index=False))
